# Решения: итоговый отчёт

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import deque


In [ ]:
stream = df[['order_id', 'is_late']].copy()
stack = []
for step in ['prepare', 'count', 'cluster', 'report']:
    stack.append(step)
queue = deque(stream['order_id'].tolist())
structures_ok = bool(stack.pop() == 'report' and len(queue) == len(df))
state_count = {}
for r in df[['customer_state', 'is_late']].itertuples(index=False):
    if r.customer_state not in state_count:
        state_count[r.customer_state] = {'late': 0, 'total': 0}
    state_count[r.customer_state]['total'] += 1
    state_count[r.customer_state]['late'] += int(r.is_late)
counts_ok = len(state_count) >= 3
features = ['delivery_days', 'freight_value', 'delay_days']
Xs = StandardScaler().fit_transform(df[features])
km = KMeans(n_clusters=3, random_state=54, n_init=10)
db = DBSCAN(eps=0.9, min_samples=4)
clustered = df.copy()
clustered['cluster_km'] = km.fit_predict(Xs)
clustered['cluster_db'] = db.fit_predict(Xs)
clusters_ok = clustered['cluster_km'].nunique() == 3
is_noise = clustered['cluster_db'] == -1
top_delay = clustered['delay_days'].rank(method='first', ascending=False) <= 5
anomalies = clustered[is_noise | top_delay].sort_values(['delay_days', 'freight_value'], ascending=False).head(8)
anomalies_ok = len(anomalies) >= 3
acceptance = pd.Series(
    [structures_ok, counts_ok, clusters_ok, anomalies_ok],
    index=['structures', 'counts', 'clusters', 'anomalies'],
)
REPORT = (
    f'Поток из {len(df)} заказов обработан с явным использованием stack/queue/deque и частот через dict. '
    f"KMeans выделил {clustered['cluster_km'].nunique()} сегмента доставки, DBSCAN пометил "
    f"{int((clustered['cluster_db'] == -1).sum())} шумовых наблюдений. "
    'Аномалии совпадают с высокими delay_days и freight_value, что указывает на рискованные маршруты и нагрузку хаба. '
    'Ограничение: учебный slim и ручной выбор параметров DBSCAN; выводы требуют проверки на полном Olist-срезе.'
)
READY = bool(acceptance.all())
EXEC = (
    'Сегментация показывает, что часть late-заказов концентрируется в кластерах с высоким delivery_days и freight_value. '
    'Рекомендуем отдельный мониторинг этих сегментов и ранний буфер для потенциальных опозданий.'
)
LIMITS = (
    'DBSCAN чувствителен к масштабу и eps, а KMeans фиксирует число кластеров заранее. '
    'Поэтому результаты интерпретируем как рабочую гипотезу, а не финальную истину.'
)
NEXT = (
    'Следующий шаг: повторить анализ на большем real Olist-срезе, добавить географические признаки и формальный протокол оценки.'
)
print(clustered[['cluster_km', 'cluster_db']].head())
print(anomalies[['order_id', 'delay_days', 'freight_value']])
print(acceptance)
print('READY=', READY)
print(REPORT)